In [ ]:
# Install imbalanced-learn for SMOTE
import subprocess
subprocess.check_call(['.venv\\Scripts\\pip.exe', 'install', '-q', 'imbalanced-learn'])
print("SMOTE library installed!")

In [ ]:
# IMPORT LIBRARIES

import sys, os, glob, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc as sk_auc
)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

print("="*80)
print("RETRAINING WITH AGGRESSIVE SMOTE (50%) + HEAVY CLASS WEIGHTS")
print("="*80)

In [ ]:
# CONFIG

CSV_FOLDER   = './MachineLearningCVE'  
LABEL_COL    = 'Label'
N_FOLDS      = 10
TEST_SIZE    = 0.30        
RANDOM_STATE = 42
OUTPUT_DIR   = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import sklearn
print(f'Scikit-learn : {sklearn.__version__}')
print(f'LightGBM     : {lgb.__version__}')
print(f'Pandas       : {pd.__version__}')

In [ ]:
# LOAD DATA

csv_files = sorted(glob.glob(os.path.join(CSV_FOLDER, '*.csv')))
if not csv_files:
    raise FileNotFoundError(f'No CSV files in {CSV_FOLDER}')

dfs = []
total_rows_read = 0
SAMPLE_ROWS = 5000000

for f in csv_files:
    print(f'  Loading {os.path.basename(f)}...', end='')
    chunks = []
    rows_from_file = 0
    
    for chunk in pd.read_csv(f, chunksize=25000, low_memory=False, encoding='latin-1'):
        rows_from_file += len(chunk)
        total_rows_read += len(chunk)
        chunks.append(chunk)
        if total_rows_read >= SAMPLE_ROWS:
            break
    
    if chunks:
        file_df = pd.concat(chunks, ignore_index=True)
        dfs.append(file_df)
        print(f' OK - {len(file_df):,} rows')
    
    if total_rows_read >= SAMPLE_ROWS:
        print(f'  Total rows limit ({SAMPLE_ROWS:,}) reached')
        break

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows loaded: {len(df):,}')
del dfs, chunks; gc.collect()
print('OK')

In [ ]:
# CLEAN DATA

df.columns = df.columns.str.strip()

drop_cols = [c for c in ['Flow ID','Source IP','Destination IP','Timestamp',
                          'Src IP','Dst IP','src_ip','dst_ip'] if c in df.columns]
if drop_cols:
    df.drop(columns=drop_cols, inplace=True)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(axis=1, thresh=int(len(df) * 0.5), inplace=True)  
df.dropna(axis=0, inplace=True)                               

before = len(df)
df.drop_duplicates(inplace=True)
print(f'Dedup removed {before - len(df):,} rows')
print(f'Shape after cleaning: {df.shape}')

In [ ]:
# ENCODE & SCALE

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df[LABEL_COL].astype(str))
class_names = le.classes_
n_classes   = len(class_names)
print(f'\nClasses ({n_classes})')

X = df.drop(columns=[LABEL_COL, 'label_enc']).select_dtypes(include=[np.number])
y = df['label_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

scaler  = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

del df; gc.collect()
print('OK')

In [ ]:
# AGGRESSIVE SMOTE: 50% OF BENIGN (vs 10% before)
print("\nApplying AGGRESSIVE SMOTE (sampling_strategy=0.5)...")
print(f'Before SMOTE: {X_train_scaled.shape}')

smote = SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
X_train_smote = pd.DataFrame(X_train_smote, columns=X.columns)

print(f'After SMOTE: {X_train_smote.shape}')

# Show class distribution change
print("\nClass distribution BEFORE SMOTE:")
print(y_train.value_counts().sort_index())
print("\nClass distribution AFTER SMOTE (50% oversampling):")
print(y_train_smote.value_counts().sort_index())

In [ ]:
# COMPUTE HEAVY CLASS WEIGHTS
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_smote), y=y_train_smote)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print("\nClass weights (balanced):")
for cls_idx, weight in sorted(class_weight_dict.items()):
    cls_name = class_names[cls_idx]
    print(f"  {cls_name:30s}: {weight:.4f}")

In [ ]:
# TRAIN BASE LEARNERS WITH CLASS WEIGHTS
print("\nTraining base learners with SMOTE data and class weights...")

base_learners = {}

# LightGBM
print("  Training LightGBM...")
lgbm_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    verbose=-1
)
lgbm_model.fit(X_train_smote, y_train_smote)
base_learners['lgbm'] = lgbm_model
print(f"    Accuracy: {lgbm_model.score(X_test_scaled, y_test):.4f}")

# Bagging with Decision Trees
print("  Training Bagging (DecisionTree)...")
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=7),
    n_estimators=50,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
bagging_model.fit(X_train_smote, y_train_smote)
base_learners['bagging'] = bagging_model
print(f"    Accuracy: {bagging_model.score(X_test_scaled, y_test):.4f}")

print("\nBase learners trained OK")

In [ ]:
# CREATE META-FEATURES FOR STACKING
print("\nGenerating meta-features...")

# Train set meta-features
meta_features_train = np.zeros((X_train_smote.shape[0], len(base_learners) * n_classes))
for i, (name, model) in enumerate(base_learners.items()):
    proba = model.predict_proba(X_train_smote)
    meta_features_train[:, i*n_classes:(i+1)*n_classes] = proba

# Test set meta-features
meta_features_test = np.zeros((X_test_scaled.shape[0], len(base_learners) * n_classes))
for i, (name, model) in enumerate(base_learners.items()):
    proba = model.predict_proba(X_test_scaled)
    meta_features_test[:, i*n_classes:(i+1)*n_classes] = proba

print(f"Meta-feature matrix: Train {meta_features_train.shape} | Test {meta_features_test.shape}")

In [ ]:
# TRAIN META-LEARNER (Logistic Regression with class weights)
print("\nTraining meta-learner with HEAVY class weights...")

meta_learner = LogisticRegression(
    max_iter=500,
    class_weight='balanced',  # Heavy balancing
    random_state=RANDOM_STATE,
    n_jobs=-1
)
meta_learner.fit(meta_features_train, y_train_smote)

print("Meta-learner trained OK")

# Evaluate ensemble
y_pred = meta_learner.predict(meta_features_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nEnsemble Accuracy on Test Set: {accuracy:.4f}")

In [ ]:
# SAVE MODELS
print("\nSaving trained models...")

joblib.dump(meta_learner, 'saved_model/meta_learner.pkl')
joblib.dump(scaler, 'saved_model/scaler.pkl')
joblib.dump(le, 'saved_model/label_encoder.pkl')
joblib.dump(base_learners, 'saved_model/base_learners.pkl')

print("  meta_learner.pkl saved")
print("  scaler.pkl saved")
print("  label_encoder.pkl saved")
print("  base_learners.pkl saved")
print("\nAll models saved OK")

In [ ]:
# DETAILED EVALUATION
print("\n" + "="*80)
print("DETAILED EVALUATION ON TEST SET")
print("="*80)

y_pred = meta_learner.predict(meta_features_test)
y_pred_proba = meta_learner.predict_proba(meta_features_test)

print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix saved")

In [ ]:
# VISUALIZE CONFUSION MATRIX
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Raw counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix - Raw Counts')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=class_names)
disp_norm.plot(ax=axes[1], cmap='RdYlGn', values_format='.1%')
axes[1].set_title('Confusion Matrix - Normalized (%)')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrix_aggressive_smote.png', dpi=120, bbox_inches='tight')
print("Confusion matrix visualization saved")
plt.close()

In [ ]:
# SUMMARY
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"\nTraining approach: AGGRESSIVE SMOTE (50%) + Heavy Class Weights")
print(f"  - SMOTE oversampling: 50% of majority class (vs 10% before)")
print(f"  - Class weights: Balanced on all models")
print(f"  - Base learners: LightGBM + Bagging")
print(f"  - Meta-learner: Logistic Regression with balanced weights")
print(f"\nResults:")
print(f"  Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"  Test set size: {len(y_test):,} samples")
print(f"\nModels saved to ./saved_model/")
print(f"Next: Test on domain-aware synthetic data to validate generalization")